In [1]:
import requests   # e "browserul" codului; folosim pentru a trimite cereri HTTP si a primi raspunsul
from bs4 import BeautifulSoup  # instrument de parsing; BS transforma html brut intr un format lizibil pt Python
from urllib.parse import urljoin, urlparse
#urljoin: lipeste fragmentele de link. De ex. daca esti pe google.com si gasesti un link catre /images,
#         il transforma automat in linkul complet: https://google.com/images
#urlparse: exact opusul; descompune un URL in bucatele (protocol, domeniu, cale), ceea ce ajuta la
#          a identifica exact de unde vine un link
import io
import PyPDF2   # pentru a citi si din pdf-uri, nu doar text html


visited = set()
MAX_DEPTH = 3

# Verifica daca linkul "link" gasit pe o alta pagina apartine aceluiasi site de baza, cel de Admitere UBB,
# sau daca este un link extern care te scoate de pe site.
# Literalmente: descompune ambele linkuri si le compara componenta de domeniu (admitere.ubbcluj.ro)
def is_internal(link, base):
    return urlparse(link).netloc == urlparse(base).netloc


def extract_text(html):
    soup = BeautifulSoup(html, "html.parser") #transforma textul brut intr-un obiect structurat (un "copac" de elemente)

    # Nu avem nevoie de textul din interiorul tagurilor urmatoare, deci le stergem cu totul
    for tag in soup(["script", "style", "nav", "footer"]): 
        tag.decompose()

    # get_text ne da "Hello!" in loc de "<p>Hello!</p>"
    return soup.get_text(separator=" ", strip=True)

def extract_text_from_pdf(content):
    pdf_file = io.BytesIO(content)  # transforma bytes in format "fisier in memorie"
    reader = PyPDF2.PdfReader(pdf_file)

    text = ""
    for page in reader.pages:
        text += page.extract_text() + " "

    return text 


def extract_links(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    links = []

    # Gaseste sectiunea principala a paginii pt a evita linkuri irelevante
    main_content = soup.find("div", class_="entry clearfix")

    if not main_content:
        return []

    for a in main_content.find_all("a", href=True): # cauta toate linkurile (eticheta <a>)
        link = urljoin(base_url, a["href"]) # pentru cazul ca e link relativ; daca in pagina scrie doar <a href="contact.html">, il transforma in https://admitere.ubbcluj.ro/contact.html
        if is_internal(link, base_url):
            links.append(link)

    return links


def crawl(url, depth=0):
    # sa nu accesam prea multe sublinkuri & sa nu le repetam pe aceleasi
    if depth > MAX_DEPTH or url in visited:
        return []

    visited.add(url)

    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/119.0.0.0"} # ca sa nu ne blocheze serverul crezand ca suntem un bot
        response = requests.get(url, headers=headers, timeout=10, stream=True)  # stream=True: descarca intai informatiile despre fisier si decide dupa daca l si descarca pt a i extrage textul

        # --VERIFICARE: Vrem ori HTML, ori pdf
        content_type = response.headers.get('Content-Type', '').lower()

        # Cazul 1: Este pagina HTML
        if 'text/html' in content_type:
            print('Vizitez url: ',url)
            html = response.text        # textul insusi, cu taguri cu tot
            text = extract_text(html)   # textul fara taguri si filtrat
            results = [text]
            for link in extract_links(html, url):
                results += crawl(link, depth + 1)

            return results
        
        # # Cazul 2: Este pdf
        elif 'application/pdf' in content_type:
            print(f"Extrag din pdf: ", url)
            text = extract_text_from_pdf(response.content)  # folosim .content (bytes), nu text
            return [text]

        # Cazul 3: Orice altceva, ignoram
        else:
            return []
        # --
        
    except Exception as e:
        print(f"Eroare la {url}: {e}")
        return []

    

In [2]:
docs = crawl("https://www.cs.ubbcluj.ro/admitere/nivel-licenta/admiterea-la-facultatea-de-matematica-si-informatica-nivel-licenta/")

Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/admiterea-la-facultatea-de-matematica-si-informatica-nivel-licenta/
Extrag din pdf:  https://www.cs.ubbcluj.ro/wp-content/uploads/Regulament-admitere-2026-UBB.pdf
Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/tematica-admitere-facultate/
Extrag din pdf:  https://www.cs.ubbcluj.ro/wp-content/uploads/anexa3a-Regulament-admitere-licenta-2026.pdf
Extrag din pdf:  https://www.cs.ubbcluj.ro/wp-content/uploads/anexa3b-Regulament-admitere-licenta-2026.pdf
Extrag din pdf:  https://www.cs.ubbcluj.ro/wp-content/uploads/anexa2-Regulament-admitere-licenta-2026.pdf
Extrag din pdf:  https://www.cs.ubbcluj.ro/wp-content/uploads/anexa1-Regulament-admitere-licenta-2026.pdf
Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/taxe-admitere-facultate-nivel-licenta/
Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/calendarul-concursului-de-admitere-nivel-licenta/


In [3]:
print(docs)

['Admiterea la Facultatea de Matematică și Informatică, nivel licență, 2026 | Facultatea de Matematică și Informatică Studenţi Anunţuri studenţi Documente pentru studenţi Taxe Plata on-line a taxelor Termenele de plată a taxelor de şcolarizare Taxe de şcolarizare pe universitate Anunţuri taxe Lista studenţilor Tutoriat şi consultaţii Lista tutorilor si orarul de tutoriat pentru anul universitar 2025/2026 Atribuţiile tutorilor Orarul consultaţiilor Practică Practică nivel licență Informaţii utile practică nivel licență Anunţuri practică nivel licență Practică nivel master Informaţii utile practică nivel master Anunțuri practică nivel master Regulament practică Burse, cazare, tabere Mobilităţi internaţionale Anunţuri burse Regulamentul privind acordarea burselor Anunţuri cazare Anunţuri tabere Examen de licenţă şi disertaţie Calendar, reglementări şi acte necesare Anunţuri licenţă şi disertaţie Manuale, subiecte și bareme pentru examenul de licenţă Manifestări științifice studențești Inf

In [35]:
# CHUNKING

def chunk_text(text, size=500):
    words = text.split()
    return [
        " ".join(words[i:i+size])
        for i in range(0, len(words), size)
    ]

chunks = []
for doc in docs:
    chunks.extend(chunk_text(doc))

In [ ]:
# EMBEDDING

from sentence_transformers import SentenceTransformer
import faiss  # baza de date specializata in viteza. Daca ai 1 milion de fragmente de text, FAISS (Facebook AI Similarity Search) gaseste instant care dintre ele seamana cu intrebarea ta
import numpy as np  # FAISS lucreaza doar cu structuri de date de tip numpy

if model not in globals():   # daca nu am definit pana acum model
    model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

# Creeaza un index (memorie rapida) care masoara distanta "L2" (distanta euclidiana, ca o linie dreapta pe harta) intre vectori
index = faiss.IndexFlatL2(len(embeddings[0]))  # len(embeddings[0]) -> ii spune indexului cat de lung este un singur vector (pentru acest model in specific lungimea fiind de 384 de numere)
index.add(np.array(embeddings))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8149.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# SAVE
np.save("chunks.npy", chunks)
faiss.write_index(index, "index.faiss")

In [ ]:
# LOAD
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np


chunks = np.load("../chunks.npy", allow_pickle=True)
index = faiss.read_index("../index.faiss")

model = SentenceTransformer("all-MiniLM-L6-v2")


c:\Users\phili\Personal\University\Licenta\Proiect\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4641.86it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
# RETRIEVAL (la query)

# Ia cele 5 cele mai relevante chunkuri pentru queryul userului
def retrieve(query, k=5):
    q_emb = model.encode([query])
    distances, indices = index.search(np.array(q_emb), k)

    return [chunks[i] for i in indices[0]]

In [5]:
import requests

def ask_llm(question, context_chunks):
    context = "\n\n".join(context_chunks)

    prompt = f"""
You answer questions about university admission.

Use ONLY the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    response = requests.post(
        "http://localhost:1234/v1/chat/completions",
        json={
            "model": "google/gemma-4-26b-a4b",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2
        }
    )


    data = response.json()
    
    # Verificăm dacă a returnat o eroare în loc de "choices"
    if "choices" not in data:
        print("Eroare primită de la LM Studio:", data)
        return "Eroare la generarea răspunsului."

    return data["choices"][0]["message"]["content"]


# NOTES
# -> am incercat si cu prompt pe romana, a dat exact acelasi raspuns

In [4]:
def answer_question(query):
    relevant_chunks = retrieve(query)
    answer = ask_llm(query, relevant_chunks)
    return answer

In [ ]:
# MODEL: Qwen 4B (18s, ~900 tokens)
print(answer_question("Care sunt actele necesare pentru admitere?"))

Pentru admitere, candidații trebuie să depună următoarele acte:

- O declarație pe proprie răspundere (Anexa 1) care să arate traiectoria școlară din învățământul universitar românesc de stat;
- Adeverința care să ateste numărul de ani bugetați sau cu taxă urmați în sistemul universitar românesc și care să arate că au achitat la zi taxele universitare și nu au debite față de Universitate;
- Diplomă de licență sau diplomă echivalentă (scanată fata verso) pentru candidații care doresc să urmeze o a doua facultate;
- Adeverințe și acte doveditoare care să ateste îndeplinirea condițiilor pentru scutire parțială sau totală de taxă de înscriere și/sau a taxei de procesare (pentru copiii personalului didactic, copiii orfani, copiii din familii monoparentale, copiii din centrele de plasament, persoanele la risc de a pierde capacitatea de a satisfăce nevoile zilnice etc.);
- Diploma cu premiul obținut sau diploma de participare pentru candidații care beneficiază de facilitățile acordate olimpic

In [ ]:
# MODEL: Mistral 7B (38s, ~1400 tokens)
print(answer_question("Care sunt actele necesare pentru admitere?"))

 Actele necesare pentru admitere la Universitatea Babeș-Bolyai sunt următoarele:

1. Declarație pe proprie răspundere (Anexa 1) care rezultă traiectoria școlară din învățământul universitar românesc de stat.
2. Adeverința care ateste numărul de ani bugetați sau cu taxă urmați în sistemul universitar românesc și din care să rezulte faptul că au achitat la zi taxele universitare și nu au debite față de Universitate.
3. Diploma de licență sau diplomă echivalentă pentru candidații care doresc să urmeze o a doua facultate – scanată fata verso.
4. Adeverințe și acte doveditoare din care să rezulte îndeplinirea condițiilor pentru scutire parțială sau totală de taxă de înscriere și/sau a taxei de procesare (copiii personalului didactic, didactic auxiliar și nedidactic aflat activitate sau pensionat din instituții, personalul didactic, didactic auxiliar și nedidactic aflat activitate sau pensionat din UBB, copiii orfani de unul sau ambii părinți, copiii proveniți din familii monoparentale, copi

In [ ]:
# MODEL: Mistral 12B (38s, ~500 tokens)
print(answer_question("Care sunt actele necesare pentru admitere?"))

Pentru admiterea la programele de studii de licență și master în Universitatea Babeș-Bolyai, candidații trebuie să depună o declarație pe proprie răspundere ( Anexa 1 ) din care să rezulte traiectoria școlară din învățământul universitar românesc de stat. Candidații care sunt studenți sau foști studenți și care se înscriu la concurs ca să urmeze o a doua facultate: Adeverință/adeverințe care să ateste numărul de ani bugetați sau cu taxă urmați în sistemul universitar românesc și din care să rezulte faptul că au achitat la zi taxele universitare și nu au debite față de Universitate; Diplomă de licență sau diplomă echivalentă pentru candidații care doresc să urmeze o a doua facultate – scanata fata verso. Adeverință și acte doveditoare din care să rezulte îndeplinirea condițiilor pentru scutire parțială sau totală de taxă de înscriere și/sau a taxei de procesare (copiii personalului didactic, didactic auxiliar și nedidactic aflat activitate sau pensionat din instituții, personalul didact

In [ ]:
# MODEL: Qwen 30B (6m 37s)
print(answer_question("Care sunt actele necesare pentru admitere?"))

Pentru admitere, candidații trebuie să depună următoarele acte:

1. O declarație pe proprie răspundere (Anexa 1), care să evidențieze traiectoria școlară din învățământul universitar românesc de stat.  
2. Pentru studenți sau foști studenți care se înscriu la o a doua facultate:  
   - Adeverință/adeverințe care să ateste numărul de ani bugetați sau cu taxă urmați în sistemul universitar românesc, din care să rezulte că au achitat la zi taxele universitare și nu au debite față de Universitate.  
   - Diplomă de licență sau diplomă echivalentă (scanată, fără verso).  
3. Adeverință și acte doveditoare pentru scutire parțială sau totală de taxă de înscriere și/sau taxă de procesare (conform Anexei 2), în cazul în care se aplică (ex: copii ai personalului didactic, persoane cu dizabilități, copii orfani, etc.). Adeverința trebuie să fie datată în perioada înscrierilor sau înainte.  
4. Diploma cu premiul obținut sau diploma de participare, pentru candidații care beneficiază de facilități 

In [ ]:
print(answer_question("Cat dureaza admiterea?"))  

# LLM-as-a-judge poate sa converteasca anumite cuvinte cheie pentru subagent, ca acesta nu le intelege altfel;
# student scrie "admitere" ==> judge schimba in "proba scrisa tip grila"
# student scrie intrebare prea lunga ==> cuvinte cheie

Contextul nu oferă informații despre durata admiterii. Deci, nu se poate răspunde cu bază în context.


In [41]:
print(answer_question("Care e durata la proba scrisa tip grila"))

Durata probei scrise tip grilă este de 3 ore.


In [ ]:
# MODEL: Mistral 7B (6s) -> NU a raspuns
print(answer_question("Cat dureaza admiterea?"))  

 The duration of the admission is not specified in the context provided. However, it mentions that the period of the admission will be determined by the University Babeș-Bolyai. For more specific information, you should check the university's official website or contact their admissions office.


In [ ]:
# same as previous cell
print(answer_question("Cat dureaza examenul de admitere?"))

 The duration of the admission exam is not specified in the context provided. However, it's mentioned that there are sessions in July/September 2026 for the Faculty of Mathematics and Informatics, but no specific timeframe for each session is given. For more detailed information about the exam schedule, you can refer to the link provided: https://www.cs.ubbcluj.ro/admitere/nivel-licenta/calendarul-concursului-de-admitere-nivel-licenta/.


In [ ]:
# same as previous cell, raspunde bine doar cand e ca pe site
print(answer_question("Cat dureaza proba scrisa tip grila?"))

 The written test, or proba scrisă tip grilă, lasts for 3 hours.


In [ ]:
# MODEL: Mistral 12B (12s) -> a raspuns bine, s a descurcat fara format
print(answer_question("Cat dureaza admiterea?"))  

The duration of the admission process is not explicitly stated in the provided context. However, it mentions that the period for the admission exam will be determined by the University of Babeș-Bolyai. The exam itself lasts 3 hours, and the results are expected to be announced after the exam. Additionally, there are provisions for candidates who may benefit from certain facilities or categories (A, B) throughout the year following their graduation from high school.


In [ ]:
# same as previous cell, totusi mai greseste...
print(answer_question("Cat dureaza examenul de admitere?"))

The duration of the admission exam is not explicitly stated in the provided context.


In [47]:
# same as previous cell, intrebari prea lungi pot incurca
print(answer_question("Cat dureaza proba scrisa tip grila care trebuie sa o dau ca sa intru?"))

The duration of the written exam in the form of a grid that you need to submit for admission is not explicitly stated in the provided context. However, it is mentioned that the media de admitere (admission average) is calculated as follows: 1/2 * 10 + 1/2 * nota la proba scrisă de concurs (note at the written exam of the contest). This suggests that the written exam is an important part of the admission process, but it does not specify how long it lasts. You should check with the specific faculty or department offering the program you are interested in for more information on the duration of the written exam.


In [ ]:
# same as previous cell, as usual se descurca daca i dai la fix
print(answer_question("Cat dureaza proba scrisa tip grila?"))

Proba scrisă tip grilă are durata de 3 ore.


In [ ]:
# MODEL: Mistral 12B 
print(answer_question("Ce documente am nevoie pentru admitere? Unde le depun? Si cat dureaza proba scrisa tip grila?"))

# Notite: 
# - prima rulare: (44s) foarte bine raspuns (partea cu "documente necesare" e mai scurta decat de obicei)
# - alta rulare: (15s) haotic - "Document necesar: doar diploma de bacalaureat & foaia matricola"
# - alte rulari: a mai raspuns bine ca prima data, dar variaza numarul de documente (ceva sigur tot omite)
# - cand am schimbat prima intrebare la "Care sunt actele necesare pentru admitere?", a enumerat un pic mai multe documente, dar
#   a raspuns mai rau la celelalte intrebari ("examenul dureaza 4 ore")
# - la a 2 a intrebare tot variaza intre "Centrul de Cooperari" si "Platforma de Admitere"

Pentru admiterea la Facultatea de Matematică și Informatică, veți avea nevoie de urmatoarele documente:

1. Diploma de bacalaureat si foaia matricola în original
2. Dovada echivalarii studiilor (pentru beneficiari ai Acordului privind retragerea Regatului Unit al Marii Britanii și Irlandei de Nord din Uniunea Europeana și din Comunitatea Europeană a Energiei Atomice 2019/C 384 I/01)
3. Diploma cu premiul obținut sau diploma de participare pentru candidații care beneficiază de facilitățile acordate olimpicilor sau participanților la concursuri.
4. Adeverință/adeverințe care să ateste numărul de ani bugetați sau cu taxă urmați în sistemul universitar românesc și din care să rezulte faptul că au achitat la zi taxele universitare și nu au debite față de Universitate (pentru candidații care doresc să urmeze o a doua facultate).
5. Diplomă de licență sau diplomă echivalentă pentru candidații care doresc să urmeze o a doua facultate – scanata fata verso.
6. Adeverință și acte doveditoare din 

In [6]:
# Gemma4 26B
print(answer_question("Care sunt actele necesare pentru admitere?"))

Eroare primită de la LM Studio: {'error': 'Model unloaded.'}
Eroare la generarea răspunsului.
